In [6]:
import pandas as pd
import numpy as np
import pmdarima as pm
import warnings

# Suppress warnings from statsmodels, which pmdarima uses
warnings.filterwarnings("ignore")

# --- 1. Configuration ---
input_file = '../../data/data_with_features.csv'
output_file = '../../data/syntheticData/synthetic_data_sarima.csv'
target_column = 'cumulative_mite_count'
group_column = 'treatment_id'

# --- 2. Load the Original Data ---
try:
    df = pd.read_csv(input_file)
    print(f"✅ Successfully loaded original data from '{input_file}'. Shape: {df.shape}")
except FileNotFoundError:
    print(f"❌ Error: The file '{input_file}' was not found.")
    exit()

# --- 3. Grouped SARIMA Generation ---
print(f"\n⚙️  Starting grouped SARIMA generation process using '{group_column}'...")

all_synthetic_groups = []
grouped_data = df.groupby(group_column)
num_groups = len(grouped_data)
print(f"   Found {num_groups} unique time series to process.")

# Process each group (each independent time series)
for i, (group_name, group_df) in enumerate(grouped_data):
    print(f"   Processing group {i+1}/{num_groups} (ID: {group_name})... ", end="")
    
    # Reset index to ensure iloc works as expected
    group_df = group_df.reset_index(drop=True)
    original_series = group_df[target_column]
    
    # A model needs a minimum number of data points to be fitted
    if len(original_series) < 20:
        print("Series too short, copying original data.")
        all_synthetic_groups.append(group_df)
        continue

    try:
        # auto_arima will automatically find the best (p,d,q)(P,D,Q,m) model
        # We set m=7 to check for weekly seasonality, a common pattern in daily data.
        sarima_model = pm.auto_arima(original_series,
                                     start_p=1, start_q=1,
                                     test='adf',       # use adftest to find optimal 'd'
                                     max_p=3, max_q=3, # maximum p and q
                                     m=7,              # frequency of series (7 for daily/weekly)
                                     d=None,           # let model determine 'd'
                                     seasonal=True,    # Seasonality
                                     start_P=0, 
                                     D=1, 
                                     trace=False,
                                     error_action='ignore',  
                                     suppress_warnings=True, 
                                     stepwise=True)

        # Generate a new series of the same length
        # We use predict_in_sample which is better for simulation
        synthetic_series = sarima_model.predict_in_sample()
        
        # --- CRITICAL: Enforce Constraints ---
        # 1. Ensure the series is non-negative
        synthetic_series[synthetic_series < 0] = 0
        # 2. Enforce the non-decreasing (monotonic) constraint for the cumulative count
        synthetic_series = np.maximum.accumulate(synthetic_series)

        # Create a new DataFrame for the synthetic group
        synthetic_group_df = group_df.copy()
        synthetic_group_df[target_column] = synthetic_series
        all_synthetic_groups.append(synthetic_group_df)
        print("Model fitted and data generated.")

    except Exception as e:
        # If auto_arima fails for any reason, fall back to copying the original data
        print(f"Model fitting failed ({e}), copying original data.")
        all_synthetic_groups.append(group_df)

# Combine all the processed synthetic groups back into one DataFrame
final_synthetic_df = pd.concat(all_synthetic_groups, ignore_index=True)

# --- 4. Save the Synthetic Data ---
final_synthetic_df.to_csv(output_file, index=False)

print("\n✅ Process complete!")
print(f"   Synthetic data generated via SARIMA and saved to '{output_file}'.")
print(f"   Final dataset shape: {final_synthetic_df.shape}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject